#Predicting Customer Lifetime Value with Multiple Regression


---



##Overview
- Objective: This assignment focuses on applying Multiple Linear Regression (MLR) to a common business problem: predicting Customer Lifetime Value (CLV). You will use Python and relevant libraries to build, interpret, and evaluate an MLR model using synthetic customer data.

- Collaboration Policy: You may discuss concepts with classmates, but the code and answers you submit must be your own individual work.

##Required Libraries

In [ ]:
#import relevant libaries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

#set a consistent plot style
sns.set_style("whitegrid")
print("Libraries imported successfully.")

Libraries imported successfully.


#Predicting Customer Lifetime Value (CLV)
##Scenario:
An e-commerce company wants to predict the potential lifetime value (CLV) of its customers. They believe CLV is influenced by several numeric factors: the customer's average purchase value, how frequently they purchase, and how long they’ve been a customer. Your task is to build an MLR model to predict CLV based on these factors.
##Data:
The starter code below generates synthetic data for 500 customers with the following numeric columns:
- AvgPurchaseValue: Average amount spent per order.
- PurchaseFrequency: Average number of purchases made per year.
- CustomerTenure: Number of years the customer has been active.
- CLV: Predicted Customer Lifetime Value - This is your target variable.




---


##Tasks:


1. Run the starter code to generate the df_clv DataFrame and briefly examine the first few rows (.head()) and basic statistics (.describe()).

In [ ]:
'''
Task #1
'''
# Starter Code: Generate Synthetic CLV Data
np.random.seed(110)
n_customers = 500

# Generate predictor variables
avg_purchase_value = np.random.normal(75, 25, n_customers) # Avg $75, std $25
purchase_frequency = np.random.normal(5, 2, n_customers)   # Avg 5 purchases/yr, std 2
customer_tenure = np.random.uniform(0.5, 10, n_customers)  # Tenure between 0.5 and 10 years

# Ensure non-negative values where logical
avg_purchase_value = np.maximum(avg_purchase_value, 10)
purchase_frequency = np.maximum(purchase_frequency, 0.5)

# Generate CLV based on a linear relationship + noise
# CLV = Base + (Effect of AvgPurchase * Freq * Tenure) - somewhat multiplicative effect logic + noise
# Let's simplify to additive for clarity in MLR interpretation for this assignment:
# CLV = Base + Effect(AvgPurchase) + Effect(Freq) + Effect(Tenure) + Noise

clv = 150 + 10 * avg_purchase_value + 80 * purchase_frequency + 50 * customer_tenure + np.random.normal(0, 250, n_customers)
clv = np.maximum(clv, 50) # Ensure minimum CLV

# Create DataFrame
df_clv = pd.DataFrame({
    'AvgPurchaseValue': avg_purchase_value,
    'PurchaseFrequency': purchase_frequency,
    'CustomerTenure': customer_tenure,
    'CLV': clv
})

print("Synthetic Customer Data Head:")
print(df_clv.head())
print("\nData Description:")
print(df_clv.describe())

Synthetic Customer Data Head:
   AvgPurchaseValue  PurchaseFrequency  CustomerTenure          CLV
0         83.214928           2.649825        2.762106  1503.484718
1         55.095036           3.438132        7.472734  1492.435762
2        110.078096           3.054568        9.820457  1367.956073
3         36.305181           3.581917        6.321292  1010.988008
4        104.168257           3.359105        8.689112  1997.209326

Data Description:
       AvgPurchaseValue  PurchaseFrequency  CustomerTenure          CLV
count        500.000000         500.000000      500.000000   500.000000
mean          75.546159           5.005602        5.229268  1561.155591
std           24.889128           1.954198        2.792919   393.090770
min           10.000000           0.500000        0.510736   401.844341
25%           58.085492           3.722779        2.783082  1281.604601
50%           76.024690           5.031070        5.120830  1571.990603
75%           93.576606           6.313

2. Define your features X (should include AvgPurchaseValue, PurchaseFrequency, CustomerTenure) and your target y (CLV).

In [ ]:
#define X (features) and Y (Target)
X_clv = df_clv[['AvgPurchaseValue', 'PurchaseFrequency', 'CustomerTenure']]
y_clv = df_clv['CLV']

#print shapes
print("\nShape of X_conv:", X_clv.shape)
print("Shape of y_conv:", y_clv.shape)


Shape of X_conv: (500, 3)
Shape of y_conv: (500,)


3. Split the data into a training set (75% of data) and a test set (25% of data). Use random_state=110 for reproducibility.


In [ ]:
#train split 75/25
X_clv_train, X_clv_test, y_clv_train, y_clv_test = train_test_split(
    X_clv, y_clv, test_size=0.25, random_state=123
)

#print training/test info
print("--- MLR Train-Test Split ---")
print("Training set size:", X_clv_train.shape[0])
print("Test set size:", X_clv_test.shape[0])

--- MLR Train-Test Split ---
Training set size: 375
Test set size: 125


4. Create an instance of the LinearRegression model.


In [ ]:
#define the regression model
clv_model = LinearRegression()

5. Train (fit) the MLR model using the training data.


In [ ]:
#train model
clv_model.fit(X_clv_train, y_clv_train)

print("CLV Model trained successfully.")

CLV Model trained successfully.


6. Print the model’s intercept (β₀) and the coefficients (β) for each predictor variable. Pair the coefficients with their feature names for clarity.


In [ ]:
#compute intercept and coefficients
clv_intercept = clv_model.intercept_
clv_coefficients = clv_model.coef_

print(f"\n--- CLV Model Coefficients ---")
print(f"Intercept (β₀): {clv_intercept:.3f}\n")

#pair coefficients to feature names
feature_names = X_clv_train.columns
clvs_coefficients = pd.DataFrame({'Feature': feature_names, 'Coefficient (β)': clv_coefficients})

#print coefficients
print(clvs_coefficients)


--- CLV Model Coefficients ---
Intercept (β₀): 176.883

             Feature  Coefficient (β)
0   AvgPurchaseValue         9.819835
1  PurchaseFrequency        78.508542
2     CustomerTenure        47.002236


7. Evaluate the model’s performance on the test set by calculating and printing the R-squared (R²) and Root Mean Squared Error (RMSE).


In [ ]:
#make predictions
y_clv_pred_test = clv_model.predict(X_clv_test)

#calculate metrics
r2_clv_test = r2_score(y_clv_test, y_clv_pred_test)
mse_clv_test = mean_squared_error(y_clv_test, y_clv_pred_test)
rmse_clv_test = np.sqrt(mse_clv_test)

#print metrics
print(f"--- CLV Model Evaluation (Test Set) ---")
print(f"R-squared (R²): {r2_clv_test:.3f}")
print(f"Root Mean Squared Error (RMSE): {rmse_clv_test:.3f}")

--- CLV Model Evaluation (Test Set) ---
R-squared (R²): 0.618
Root Mean Squared Error (RMSE): 230.126


8. Make a prediction for the CLV of a new hypothetical customer with the following profile:
- AvgPurchaseValue = $85
- PurchaseFrequency = 6 purchases/year
- CustomerTenure = 4 years Print the predicted CLV for this customer.


In [ ]:
#create prediction dataframe
customer = pd.DataFrame({
    'AvgPurchaseValue': [85],  #85 dollars
    'PurchaseFrequency': [6],         #6 purchases a year
    'CustomerTenure': [4]            #4 years
})

#predict clv for customer profile
hypo_pred = clv_model.predict(customer)

#print prediction
print(f"\n--- CLV Customer Prediction ---")
print(f"Predicted Conversion Rate for profile: ${hypo_pred[0]:.2f}")


--- CLV Customer Prediction ---
Predicted Conversion Rate for profile: $1670.63


##Interpretation Questions:
Explain the meaning of the coefficient for PurchaseFrequency in your trained model. Remember to mention the other variables in your interpretation.
Explain what the R-squared value calculated on the test set tells you about your model’s predictive capability for CLV based on these features.

- For every additional purchase in PurchaseFrequency, the Customer Lifetime Value (CLV) is predicted to increase by 78.50 percentage points, holding AvgPurchaseValue and CustomerTenure constant.
- About 61.8% of the variability in the CLV can be explained by the combination of features in X (AvgPurchaseValue, PurchaseFrequency, CustomerTenure) given by R².